# 직업 업무 설명 100개 시험 임베딩·검색

직업 100개의 `work` 전체를 각각 하나의 검색 단위로 사용합니다. 기존 Job 노드에 `work_embedding`을 저장하며 Document·Chunk 노드를 새로 만들지 않습니다. PDF·적성·전망은 이번 검색 대상이 아닙니다.

- 모델: `text-embedding-3-small`, 768차원
- 인덱스: `job_work_embedding_768`
- 검색 결과: `job_code`, `job_name`, `work` 세 필드
- 기존 청크 임베딩 100개는 입력 내용이 달라 재사용하지 않습니다.
- 셀 1~4는 읽기·검사, 셀 5~6은 유료 API 호출과 Aura 기록, 셀 7은 인덱스 생성입니다.
- 위에서 아래로 실행하세요. 생성 시점에는 실행하지 않았으며 저장된 실행 결과가 없습니다.


## 셀 1 — 이 노트북의 환경과 DB 연결
01·02의 변수에 의존하지 않습니다. 현재 노트북 폴더에서 프로젝트 루트를 찾고 `.env`를 읽습니다. 키와 비밀번호는 출력하지 않습니다.


In [1]:
import os
import json
import hashlib
import math
import time
from pathlib import Path
from dotenv import dotenv_values
from neo4j import GraphDatabase

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'data/expanded_20261108/jobs.csv').is_file()
    and (p / '.env').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('팀 프로젝트 폴더 또는 하위 폴더에서 노트북을 실행하세요.')
env = dotenv_values(PROJECT_ROOT / '.env')
username = env.get('NEO4J_USERNAME') or env.get('NEO4J_USER')
if not username or any(not env.get(k) for k in
    ('NEO4J_URI', 'NEO4J_PASSWORD', 'NEO4J_DATABASE', 'OPENAI_API_KEY')):
    raise ValueError('.env의 DB 설정과 OPENAI_API_KEY를 확인하세요.')
MODEL = 'text-embedding-3-small'
DIM = 768
INPUT_VERSION = 'job-work-v1'
INDEX = 'job_work_embedding_768'
DATABASE = env['NEO4J_DATABASE']
driver = GraphDatabase.driver(env['NEO4J_URI'],
    auth=(username, env['NEO4J_PASSWORD']), connection_timeout=10,
    max_transaction_retry_time=15)

def run_cypher(query, **params):
    with driver.session(database=DATABASE) as session:
        return [r.data() for r in session.run(query, **params)]

print('DB 연결:', run_cypher('RETURN 1 AS ok'))
print('프로젝트:', PROJECT_ROOT)


DB 연결: [{'ok': 1}]
프로젝트: c:\Users\Playdata\Desktop\mle-01-p3-team3


## 셀 2 — 전체 Job과 기존 인덱스 검사
업무 설명이 비어 있는 직업은 제외합니다. ID 누락·중복, 속성 타입과 기존 벡터 인덱스의 충돌을 검사합니다. 다른 설정의 인덱스를 삭제하지 않습니다.


In [2]:
print("대상 DB:", DATABASE)
print(run_cypher("MATCH (n) RETURN labels(n) AS labels, count(n) AS count"))

대상 DB: 2a55805a
[]


In [3]:
run_cypher("MATCH (n) DETACH DELETE n")
run_cypher("DROP INDEX career_api_chunk_768 IF EXISTS")
run_cypher("DROP INDEX job_work_embedding_768 IF EXISTS")
print(run_cypher("MATCH (n) RETURN count(n) AS nodes"))

[{'nodes': 0}]


In [4]:
print("프로젝트:", PROJECT_ROOT)
print("URI:", env["NEO4J_URI"])
print("DB:", DATABASE)

print("전체 노드:", run_cypher(
    "MATCH (n) RETURN count(n) AS count"
))
print("라벨별:", run_cypher(
    "MATCH (n) RETURN labels(n) AS labels, count(n) AS count"
))

프로젝트: c:\Users\Playdata\Desktop\mle-01-p3-team3
URI: neo4j+s://2a55805a.databases.neo4j.io
DB: 2a55805a
전체 노드: [{'count': 0}]
라벨별: []


In [5]:
print(run_cypher(
    "MATCH (j:Job) RETURN count(j) AS jobs"
))

[{'jobs': 552}]


In [6]:
job_rows = run_cypher("""
MATCH (j:Job)
RETURN j.id AS id, j.source_id AS job_code, j.name AS job_name, j.work AS work
ORDER BY id
""")
if not job_rows:
    raise ValueError('Job 노드가 없습니다. 대상 DB를 확인하세요.')
ids = [r['id'] for r in job_rows]
codes = [str(r['job_code']) if r['job_code'] is not None else None for r in job_rows]
if any(not x for x in ids) or len(set(ids)) != len(ids):
    raise ValueError('Job.id 누락 또는 중복')
if any(not x for x in codes) or len(set(codes)) != len(codes):
    raise ValueError('Job.source_id 누락 또는 중복')
eligible = []
for row in job_rows:
    row['job_code'] = str(row['job_code'])
    if row['id'] != 'job:' + row['job_code']:
        raise ValueError(f"ID 불일치: {row['id']}")
    if not isinstance(row['job_name'], str) or not row['job_name'].strip():
        raise ValueError(f"직업명 누락: {row['id']}")
    if row['work'] is not None and not isinstance(row['work'], str):
        raise ValueError(f"work가 문자열이 아님: {row['id']}")
    if row['work'] and row['work'].strip():
        eligible.append(row)
print('Job:', len(job_rows), '/ 업무 설명 있음:', len(eligible),
      '/ 빈 업무 설명:', len(job_rows) - len(eligible))

def vector_indexes():
    return run_cypher("""SHOW VECTOR INDEXES
    YIELD name, state, labelsOrTypes, properties, options
    RETURN name, state, labelsOrTypes, properties, options""")

def check_index(index):
    config = index['options'].get('indexConfig', {})
    if (index['labelsOrTypes'] != ['Job']
        or index['properties'] != ['work_embedding']
        or config.get('vector.dimensions') != DIM
        or config.get('vector.similarity_function', '').lower() != 'cosine'):
        raise ValueError('기존 업무 벡터 인덱스 설정 충돌: ' + index['name'])

for index in vector_indexes():
    if index['name'] == INDEX:
        check_index(index)
    elif index['labelsOrTypes'] == ['Job'] and index['properties'] == ['work_embedding']:
        raise ValueError('같은 속성의 다른 인덱스가 있습니다: ' + index['name'])
print('기존 인덱스 검사 완료')


Job: 552 / 업무 설명 있음: 552 / 빈 업무 설명: 0
기존 인덱스 검사 완료


In [7]:
state = run_cypher("""
MATCH (j:Job)
RETURN count(j) AS jobs,
       count(j.work_embedding) AS work_vectors
""")[0]

other = run_cypher("""
OPTIONAL MATCH (d:Document)
WITH count(d) AS documents
OPTIONAL MATCH (c:Chunk)
RETURN documents, count(c) AS chunks
""")[0]

print(state, other)

if state["jobs"] != 552 or state["work_vectors"] != 0:
    raise ValueError("기본 그래프만 적재된 초기 상태가 아닙니다.")
if other["documents"] != 0 or other["chunks"] != 0:
    raise ValueError("이전 Document·Chunk가 남아 있습니다.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `work_embedding` does not exist in database `2a55805a`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=16, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (j:Job)\nRETURN count(j) AS jobs,\n       count(j.work_embedding) AS work_vectors\n'


{'jobs': 552, 'work_vectors': 0} {'documents': 0, 'chunks': 0}


## 셀 3 — 직업 100개 선택·입력과 토큰 수 확정
같은 데이터에서는 같은 표본이 선택됩니다. 일기예보관을 포함하고 나머지는 ID 해시 순으로 선택합니다. 업무 원문은 변경하지 않으며, 입력 한도 초과 항목을 잘라 넣지 않습니다. 토큰 수는 예상 비용 계산용이며 실제 청구액이 아닙니다.


In [8]:
import tiktoken
if len(eligible) < 100:
    raise ValueError('업무 설명이 있는 직업이 100개 미만입니다.')
priority = [r for r in eligible if r['id'] == 'job:5']
rest = sorted((r for r in eligible if r['id'] != 'job:5'),
    key=lambda r: hashlib.sha256(r['id'].encode()).hexdigest())
sample = [dict(r) for r in (priority + rest)[:100]]
encoding = tiktoken.encoding_for_model(MODEL)
for row in sample:
    row['input_text'] = f"직업명: {row['job_name']}\n하는 일: {row['work']}"
    row['input_hash'] = hashlib.sha256(row['input_text'].encode('utf-8')).hexdigest()
    row['tokens'] = len(encoding.encode(row['input_text']))
    if row['tokens'] > 8191:
        raise ValueError(f"입력 한도 초과: {row['id']}, {row['tokens']} 토큰")
sample_ids = [r['id'] for r in sample]
print('시험 직업:', len(sample), '/ 예상 입력 토큰:', sum(r['tokens'] for r in sample))
print('최대 입력 토큰:', max(r['tokens'] for r in sample))
print('입력 예시:\n', sample[0]['input_text'])


시험 직업: 100 / 예상 입력 토큰: 28450
최대 입력 토큰: 468
입력 예시:
 직업명: 일기예보관
하는 일: 일기예보관은 각종 기상 관측 자료의 분석을 바탕으로 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표하는 업무를 수행합니다. | 기류의 방향, 속도, 온도, 습도, 기압 등 지구 대기의 물리적 특성과 그 특성에 영향을 미치는 요인을 조사하여 일기도를 작성합니다. | 기상 관측 자료의 분석을 통해 장기, 단기 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표합니다.


## 셀 4 — 기존 업무 벡터 재사용 여부 검사
모델·차원·입력 버전·해시가 모두 같을 때만 재사용합니다. 이전 Chunk 벡터는 이 검사 대상이 아닙니다. 다른 프로세스가 Job 본문을 변경한 경우에도 중단합니다.


In [9]:
def load_states():
    rows = run_cypher("""
    MATCH (j:Job) WHERE j.id IN $ids
    RETURN j.id AS id, j.name AS name, j.work AS work,
           j.work_embedding IS NOT NULL AS ready,
           size(j.work_embedding) AS actual_dimension,
           j.work_embedding_model AS model,
           j.work_embedding_dim AS dimension,
           j.work_embedding_version AS version,
           j.work_embedding_hash AS input_hash
    """, ids=sample_ids)
    if len(rows) != 100 or len({r['id'] for r in rows}) != 100:
        raise ValueError('표본 Job 누락 또는 중복')
    return {r['id']: r for r in rows}

def reusable(row, state):
    if state['name'] != row['job_name'] or state['work'] != row['work']:
        raise ValueError('표본 선택 후 본문 변경: ' + row['id'])
    if not state['ready']:
        return False
    if (state['model'] != MODEL or state['dimension'] != DIM
        or state['actual_dimension'] != DIM or state['version'] != INPUT_VERSION
        or state['input_hash'] != row['input_hash']):
        raise ValueError('기존 업무 벡터의 설정·출처 충돌: ' + row['id'])
    return True

states = load_states()
reuse = sum(reusable(r, states[r['id']]) for r in sample)
print('기존 업무 벡터 재사용:', reuse, '/ 새로 임베딩:', 100 - reuse)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `work_embedding_dim` does not exist in database `2a55805a`. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=255>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 255, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (j:Job) WHERE j.id IN $ids\n    RETURN j.id AS id, j.name AS name, j.work AS work,\n           j.work_embedding IS NOT NULL AS ready,\n           size(j.work_embedding) AS actual_dimension,\n           j.work_embedding_model AS model,\n           j.work_embedding_dim AS dimension,\n    

기존 업무 벡터 재사용: 0 / 새로 임베딩: 100


## 셀 5 — 10개 소량 시험: 임베딩 생성·기존 Job에 저장
이 셀은 OpenAI API를 호출하고 Aura에 기록합니다. 배치 저장 건수가 다르면 해당 DB 트랜잭션을 취소합니다. 생성한 벡터는 커널 메모리에도 보관해 DB 저장 실패 후 같은 커널에서 재실행할 때 재사용합니다. 커널 종료 시 메모리 캐시는 사라집니다.


In [10]:
from openai import OpenAI
client = OpenAI(api_key=env['OPENAI_API_KEY'], timeout=60, max_retries=2)
if '_work_vector_cache' not in globals():
    _work_vector_cache = {}
if 'work_run_log' not in globals():
    work_run_log = []

def cache_key(row):
    return (MODEL, DIM, INPUT_VERSION, row['input_hash'])

def embed_and_store(rows):
    states = load_states()
    pending = [r for r in rows if not reusable(r, states[r['id']])]
    print('대상:', len(rows), '/ DB 재사용:', len(rows)-len(pending))
    for start in range(0, len(pending), 10):
        batch = pending[start:start+10]
        need_api = [r for r in batch if cache_key(r) not in _work_vector_cache]
        stage = 'embedding'
        try:
            if need_api:
                response = client.embeddings.create(model=MODEL, dimensions=DIM,
                    input=[r['input_text'] for r in need_api])
                data = sorted(response.data, key=lambda item: item.index)
                if [x.index for x in data] != list(range(len(need_api))):
                    raise ValueError('임베딩 응답 개수 또는 순서 이상')
                for row, item in zip(need_api, data):
                    vector = item.embedding
                    if len(vector) != DIM or not all(math.isfinite(x) for x in vector):
                        raise ValueError('벡터 차원 또는 수치 이상')
                    _work_vector_cache[cache_key(row)] = vector
                work_run_log.append({'stage':'api_success',
                    'ids':[r['id'] for r in need_api],
                    'tokens':response.usage.total_tokens})
            stage = 'database'
            payload = [{**r, 'vector':_work_vector_cache[cache_key(r)]} for r in batch]
            def store(tx):
                count = tx.run("""
                UNWIND $rows AS row
                MATCH (j:Job {id: row.id})
                WHERE j.name = row.job_name AND j.work = row.work
                  AND (j.work_embedding IS NULL OR
                    (j.work_embedding_model = $model
                     AND j.work_embedding_dim = $dim
                     AND j.work_embedding_version = $version
                     AND j.work_embedding_hash = row.input_hash))
                SET j.work_embedding = row.vector,
                    j.work_embedding_model = $model,
                    j.work_embedding_dim = $dim,
                    j.work_embedding_version = $version,
                    j.work_embedding_hash = row.input_hash,
                    j.work_embedding_updated_at = datetime()
                RETURN count(j) AS saved
                """, rows=payload, model=MODEL, dim=DIM, version=INPUT_VERSION).single()['saved']
                if count != len(batch):
                    raise RuntimeError('Job 누락 또는 동시 변경 감지: 배치 저장 취소')
                return count
            with driver.session(database=DATABASE) as session:
                saved = session.execute_write(store)
            work_run_log.append({'stage':'db_success','ids':[r['id'] for r in batch]})
            print('저장 완료:', saved, '건')
        except Exception as exc:
            work_run_log.append({'stage':stage+'_failed',
                'ids':[r['id'] for r in batch], 'error_type':type(exc).__name__})
            print('실패 단계:', stage, '/ 유형:', type(exc).__name__,
                  '/ ID:', [r['id'] for r in batch])
            raise

embed_and_store(sample[:10])
states = load_states()
assert all(reusable(r, states[r['id']]) for r in sample[:10])
print('소량 10개 검증 통과')


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `work_embedding_dim` does not exist in database `2a55805a`. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=255>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 255, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (j:Job) WHERE j.id IN $ids\n    RETURN j.id AS id, j.name AS name, j.work AS work,\n           j.work_embedding IS NOT NULL AS ready,\n           size(j.work_embedding) AS actual_dimension,\n           j.work_embedding_model AS model,\n           j.work_embedding_dim AS dimension,\n    

대상: 10 / DB 재사용: 0
저장 완료: 10 건
소량 10개 검증 통과


## 셀 6 — 시험 표본 100개까지 확대
앞의 10개가 정상일 때 나머지를 처리합니다. 재실행해도 DB의 일치하는 벡터는 건너뜁니다. 입력 해시·버전이 달라지면 자동 덮어쓰기하지 않습니다.


In [11]:
states = load_states()
if not all(reusable(r, states[r['id']]) for r in sample[:10]):
    raise RuntimeError('셀 5의 소량 시험을 먼저 완료하세요.')
embed_and_store(sample)
states = load_states()
assert all(reusable(r, states[r['id']]) for r in sample)
print('시험 표본 100개 모두 저장·검증 완료')
print('이 커널에서 기록한 API 입력 토큰:',
      sum(r.get('tokens', 0) for r in work_run_log))
print('실패 기록:', [r for r in work_run_log if r['stage'].endswith('_failed')])


대상: 100 / DB 재사용: 10
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
저장 완료: 10 건
시험 표본 100개 모두 저장·검증 완료
이 커널에서 기록한 API 입력 토큰: 28450
실패 기록: []


## 셀 7 — 업무 검색 전용 인덱스 생성·확인
기존 Chunk 인덱스는 사용하지 않습니다. 같은 이름 또는 같은 속성의 다른 인덱스와 충돌하면 중단합니다. 새 벡터가 저장되면 기존 인덱스에도 반영됩니다.


In [12]:
for index in vector_indexes():
    if index['name'] == INDEX:
        check_index(index)
    elif index['labelsOrTypes'] == ['Job'] and index['properties'] == ['work_embedding']:
        raise ValueError('같은 속성의 다른 인덱스: ' + index['name'])
run_cypher(f"""
CREATE VECTOR INDEX {INDEX} IF NOT EXISTS
FOR (j:Job) ON (j.work_embedding)
OPTIONS {{indexConfig: {{
 `vector.dimensions`: 768,
 `vector.similarity_function`: 'cosine'
}}}}
""")
for _ in range(20):
    index = next((r for r in vector_indexes() if r['name'] == INDEX), None)
    if index is None: raise RuntimeError('인덱스 생성 확인 실패')
    check_index(index)
    if index['state'] == 'ONLINE': break
    if index['state'] == 'FAILED': raise RuntimeError('인덱스 생성 실패')
    time.sleep(2)
else:
    raise TimeoutError('아직 준비 중입니다. 잠시 후 이 셀을 다시 실행하세요.')
print(INDEX, index['state'])
print(run_cypher("""MATCH (j:Job)
RETURN count(j) AS jobs, count(j.work_embedding) AS embedded_jobs"""))


job_work_embedding_768 ONLINE
[{'jobs': 552, 'embedded_jobs': 100}]


## 셀 8 — 의미 검색: 메타데이터 세 항목만 반환
질문도 small·768차원으로 임베딩합니다. 검색 결과는 `job_code`, `job_name`, `work`만 반환합니다. 점수·임베딩 운영 기록은 결과 메타데이터에 넣지 않습니다. 인덱스에 다른 업무 벡터가 있다면 같은 모델·설정인지 검사한 뒤 검색합니다.


In [13]:
def search_work(question, top_k=5):
    if not isinstance(question, str) or not question.strip():
        raise ValueError('질문을 입력하세요.')
    if not isinstance(top_k, int) or not 1 <= top_k <= 20:
        raise ValueError('top_k는 1~20 정수로 지정하세요.')
    if len(encoding.encode(question)) > 8191:
        raise ValueError('질문 입력 한도 초과')
    incompatible = run_cypher("""
    MATCH (j:Job) WHERE j.work_embedding IS NOT NULL
    AND (coalesce(j.work_embedding_model, '') <> $model
      OR coalesce(j.work_embedding_dim, -1) <> $dim
      OR size(j.work_embedding) <> $dim
      OR coalesce(j.work_embedding_version, '') <> $version)
    RETURN count(j) AS n
    """, model=MODEL, dim=DIM, version=INPUT_VERSION)[0]['n']
    if incompatible:
        raise ValueError('업무 인덱스에 다른 설정의 벡터가 있습니다.')
    response = client.embeddings.create(model=MODEL, dimensions=DIM, input=[question])
    vector = response.data[0].embedding
    if len(vector) != DIM: raise ValueError('질문 벡터 차원 불일치')
    return run_cypher("""
    CALL db.index.vector.queryNodes($index, $k, $vector)
    YIELD node AS j, score
    RETURN toString(j.source_id) AS job_code,
           j.name AS job_name, j.work AS work
    ORDER BY score DESC
    """, index=INDEX, k=top_k, vector=vector)

for question in ['일기예보관은 어떤 일을 하나요?',
                 '날씨 관측 자료를 분석해 예보하는 직업은?']:
    print('\n질문:', question)
    results = search_work(question, top_k=3)
    assert all(set(r) == {'job_code','job_name','work'} for r in results)
    print(json.dumps(results, ensure_ascii=False, indent=2))



질문: 일기예보관은 어떤 일을 하나요?


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    CALL db.index.vector.queryNodes($index, $k, $vector)\n    YIELD node AS j, score\n    RETURN toString(j.source_id) AS job_code,\n           j.name AS job_name, j.work AS work\n    ORDER BY score DESC\n    '


[
  {
    "job_code": "5",
    "job_name": "일기예보관",
    "work": "일기예보관은 각종 기상 관측 자료의 분석을 바탕으로 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표하는 업무를 수행합니다. | 기류의 방향, 속도, 온도, 습도, 기압 등 지구 대기의 물리적 특성과 그 특성에 영향을 미치는 요인을 조사하여 일기도를 작성합니다. | 기상 관측 자료의 분석을 통해 장기, 단기 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표합니다."
  },
  {
    "job_code": "337",
    "job_name": "촬영기사",
    "work": "촬영기사는 영화 카메라나 스튜디오 카메라 등으로 각종 물체나 대상을 촬영합니다. | 화면의 노출 상태, 촬영 대상과 카메라의 움직임, 촬영 대상과의 거리 변경 등을 고려하여 방송연출가나 촬영감독의 지시에 따라 촬영 대상을 찍습니다. | 촬영이 끝나면 촬영 일시, 장면 등을 기록하며 촬영 장비를 점검하고, 파손된 장비와 인화 장비 등을 수리합니다. | 촬영이 끝난 필름을 방송연출가 또는 편집기사와 함께 편집하기도 합니다. | 촬영 목적과 영역에 따라 영화촬영기사, 광고촬영기사, TV촬영기사, 비디오촬영기사 등으로 나뉠 수 있습니다."
  },
  {
    "job_code": "976",
    "job_name": "사진기자",
    "work": "사진기자는 신문이나 잡지 등에 게재되는 뉴스, 스포츠, 연예 등 관련 분야의 현장 보도사진을 찍습니다. | 독자의 관심과 흥미를 불러일으킬 수 있는 사건, 사고 현장 및 장소를 찾아 사건, 사고의 전개 과정이나 정보의 중요성에 따라 주요 장면을 사진 촬영합니다. | 촬영한 사진을 현상하고 편집하여 기사나 정보의 순서에 맞게 편집부에 제공합니다."
  }
]

질문: 날씨 관측 자료를 분석해 예보하는 직업은?


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    CALL db.index.vector.queryNodes($index, $k, $vector)\n    YIELD node AS j, score\n    RETURN toString(j.source_id) AS job_code,\n           j.name AS job_name, j.work AS work\n    ORDER BY score DESC\n    '


[
  {
    "job_code": "5",
    "job_name": "일기예보관",
    "work": "일기예보관은 각종 기상 관측 자료의 분석을 바탕으로 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표하는 업무를 수행합니다. | 기류의 방향, 속도, 온도, 습도, 기압 등 지구 대기의 물리적 특성과 그 특성에 영향을 미치는 요인을 조사하여 일기도를 작성합니다. | 기상 관측 자료의 분석을 통해 장기, 단기 일기예보와 기상현상에 대한 주의보, 경보, 기상 전망을 발표합니다."
  },
  {
    "job_code": "10037",
    "job_name": "생물 정보 분석가",
    "work": "생물 정보 분석가는 생물의 유전자에서 출발해 단백질, 세포, 기관, 인간 전체에서 나오는 생물학적인 정보를 수집하여 데이터로 만들고, 이를 발전시키는 작업을 합니다. | 이렇게 나온 정보를 비교 및 분석이 가능하도록 조직화하고, 조직화한 데이터를 다시 새롭게 만들거나 분석해서 연구자에게 제공하기도 합니다. | 데이터를 분석해 새로운 현상을 발견하는 일도 하는데, 이때 그 도구로 컴퓨터 프로그램이나 언어를 활용하며, 이를 개발하는 과정에도 참여합니다."
  },
  {
    "job_code": "10032",
    "job_name": "빅 데이터 전문가",
    "work": "빅 데이터 전문가는 대량의 빅 데이터로 사람들의 행동이나 시장의 변화 등을 분석하는 데 도움이 되는 정보를 제공합니다. | 구체적으로는 데이터 수집, 데이터 저장 및 분석, 데이터 시각화 등을 통한 정보 제공을 담당합니다. | 실시간으로 데이터를 수집·저장·분석하고 시각화하여 의미 있는 분석 결과를 도출하기도 합니다. | 빅 데이터와 관련된 새로운 기술, 유행 등을 수시로 파악합니다"
  }
]


## 셀 9 — 검색된 직업에서 기존 그래프 관계 확인
업무 검색 결과의 job_code로 안정적인 Job.id를 구성합니다. 관련 자격을 필수 자격으로 해석하지 않습니다. 이 조회는 업무 검색의 세 필드 반환과 별도입니다.


In [14]:
results = search_work('날씨 관측 자료를 분석해 예보하는 직업은?', top_k=1)
if results:
    job_id = 'job:' + results[0]['job_code']
    paths = run_cypher("""
    MATCH (j:Job {id: $id})-[r]->(target)
    WHERE type(r) IN ['RELATED_MAJOR','RELATED_CERTIFICATE','HAS_CORE_ABILITY']
    RETURN j.id AS job_id, type(r) AS relation, target.name AS target_name,
           r.source_file AS source_file, r.source_locator AS source_locator,
           r.source_record_json AS source_record_json
    ORDER BY relation, target_name
    """, id=job_id)
    print(json.dumps(paths, ensure_ascii=False, indent=2))
else:
    print('검색 결과가 없습니다.')


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    CALL db.index.vector.queryNodes($index, $k, $vector)\n    YIELD node AS j, score\n    RETURN toString(j.source_id) AS job_code,\n           j.name AS job_name, j.work AS work\n    ORDER BY score DESC\n    '


[
  {
    "job_id": "job:5",
    "relation": "HAS_CORE_ABILITY",
    "target_name": "수리·논리력",
    "source_file": "cache/job_5.json",
    "source_locator": null,
    "source_record_json": null
  },
  {
    "job_id": "job:5",
    "relation": "RELATED_CERTIFICATE",
    "target_name": "기상감정기사",
    "source_file": "cache/job_5.json",
    "source_locator": null,
    "source_record_json": null
  },
  {
    "job_id": "job:5",
    "relation": "RELATED_CERTIFICATE",
    "target_name": "기상기사",
    "source_file": "cache/job_5.json",
    "source_locator": null,
    "source_record_json": null
  },
  {
    "job_id": "job:5",
    "relation": "RELATED_CERTIFICATE",
    "target_name": "기상예보기술사",
    "source_file": "cache/job_5.json",
    "source_locator": null,
    "source_record_json": null
  },
  {
    "job_id": "job:5",
    "relation": "RELATED_MAJOR",
    "target_name": "대기과학과",
    "source_file": "cache/job_5.json",
    "source_locator": null,
    "source_record_json": null
  },
  {
    "job_id": "

## 실행 후 확인
- 표본 100개가 모델·차원·입력 해시 검증을 통과했는지 확인합니다.
- `job_work_embedding_768`이 ONLINE인지 확인합니다.
- 직업명 포함 질문과 직업명 없는 질문 모두 원문과 대조합니다. 100개 표본에 없는 직업까지 검색할 수 있는 상태는 아닙니다.
- 기존 청크 100개와 업무 벡터 100개는 서로 다른 데이터입니다.
- 작업을 마친 뒤 필요하면 별도 셀에서 `driver.close()`로 연결을 닫으세요. 재사용할 때는 연결 셀을 다시 실행합니다.

참고: 프로젝트 docs/CareerGraph_work_임베딩_Neo4j_인계서.md, 교과목 2/day35_벡터검색_GraphRAG/교안_01_임베딩적재_벡터전문인덱스.ipynb.
